1 替换对应字段的内容，通过客制化消息的方式，达到以下好处

1 在可见的对话中使用更**自然的消息**替代原始数据 2使用简短的确认信息替代可能很长的数据库，**减少Token的消耗**

In [1]:
from dataclasses import dataclass
from typing import Literal, TypedDict, Annotated, Union
from langchain.agents.structured_output import ProviderStrategy,ToolStrategy,AutoStrategy
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from langchain.agents import create_agent
import os
from pydantic import BaseModel, Field
from rich import print as rprint
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.tools import tool
load_dotenv(override=True)
my_api_key=os.getenv("API_KEY")
my_base_url=os.getenv("BASE_URL")
#1 初始化大模型
myModel= init_chat_model(
    model="qwen3.8-max",
    model_provider="openai",
    api_key=my_api_key,
    base_url=my_base_url,
    timeout=30,
    max_retries=3,
    reasoning_effort="none"#思考模式    #'none', 'minimal', 'low', 'medium', 'high', 'xhigh', 'max'",
)
class ContractInfo(BaseModel):
    """
    提取用户的联系方式
    """
    name:str=Field(description="用户姓名")
    email:str=Field(description="用户邮箱号")
    phone:str=Field(description="用户手机号")
#3 初始化我的agent
myAgent=create_agent(
    model=myModel,
    name="agent01",
    tools=[],
    # system_prompt="你是1个专业的天气预报小助手,当有人问你关于天气问题的时候，你会调用工具get_weather来帮你完成查询任务。"
    system_prompt=SystemMessage(content="你是1个专业的信息处理小助手,当有人问你关于信息处理问题的时候，你会帮助询问者完成信息处理任务。"),
    response_format=ToolStrategy(
        schema=ContractInfo,
        tool_message_content="已成功抽取结构信息"
    ),
)
#4 调用agent
response=myAgent.invoke(
    {
        # 严格校验，使用message少了1个s就会报错
        "messages":
            [                HumanMessage("请你提取一下该员工的联系信息：小明的邮箱是1414227322@qq.com,手机是18788483333")        ]
    }
)
rprint(response)

{
    'messages': [
        HumanMessage(
            content='请你提取一下该员工的联系信息：小明的邮箱是1414227322@qq.com,手机是18788483333',
            additional_kwargs={},
            response_metadata={},
            id='8a928ef7-0ac0-43b0-a960-5d7e404ed40b'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 66,
                    'prompt_tokens': 384,
                    'total_tokens': 450,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 0,
                        'image_tokens': None,
                        'text_tokens': 384
                    }
                },
                'model_provider': 'openai',
                'model_name': 'qwen3.8-max',
                'system_fingerprint': None,
                'id': 'chatcmpl-812c539c-8eb5-963a-9cf5-16b773cd7027',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            name='agent01',
            id='lc_run--01a0aa73-2d92-7eb0-bd47-b35888e9b722-0',
            tool_calls=[
                {
                    'name': 'ContractInfo',
                    'args': {'name': '小明', 'email': '1414227322@qq.com', 'phone': '18788483333'},
                    'id': 'call_e9ea680f72fe421d8f8d7c00',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 384,
                'output_tokens': 66,
                'total_tokens': 450,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content='已成功抽取结构信息',
            name='ContractInfo',
            id='0a6135c6-c6cb-4918-8fe4-adb4e2ab673d',
            tool_call_id='call_e9ea680f72fe421d8f8d7c00'
        )
    ],
    'structured_response': ContractInfo(name='小明', email='1414227322@qq.com', phone='18788483333')
}